# 02 — The Topological Clock

**How does the shape of the knowledge landscape change over time?**

We compute persistent homology not on the raw patent citation graph (intractable
at 20K+ nodes), but on the **knowledge space** defined by CPC subclass co-citation
patterns. Each of ~260 CPC subclasses becomes a point. The distance between two
subclasses is defined by how differently they cite other subclasses (cosine distance
on their citation vectors). Vietoris-Rips persistent homology on this ~260-point
distance matrix completes in seconds per window.

This answers a better question than the raw graph approach: *What is the shape of
the knowledge landscape, and how does it change before breakthroughs?*

- **β₀ dropping** = previously disconnected fields merging in citation space
- **β₁ appearing** = circular citation flows forming between field clusters
- **Persistence entropy** = overall topological complexity of the knowledge space

---

In [ ]:
# %% Imports and setup
import logging
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import DATA_DIR, get_logger
from src.topology import (
    compute_all_priority_pairs,
    compute_global_topology,
    PRIORITY_PAIRS,
    build_cocitation_matrix,
    cocitation_to_distance,
    compute_persistence,
    betti_numbers,
    persistence_entropy,
)
from src.metrics import cpc_mixing_rate
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, year_axis

set_style()
logger = get_logger('nb02')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

In [ ]:
# %% Load data
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

# Ensure citing_year column exists (required by new topology module)
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year

print(f'Patents: {len(patents):,}')
print(f'Citations: {len(citations):,}')
print(f'CPC mappings: {len(cpc_map):,}')
print(f'Year range: {citations["citing_year"].min()} - {citations["citing_year"].max()}')

# Clean up any stale pickle caches from old approach
stale = list((DATA_DIR / 'topology_cache').glob('sliding_*.pkl')) if (DATA_DIR / 'topology_cache').exists() else []
if stale:
    print(f'WARNING: {len(stale)} stale pickle caches found from old approach. Clearing...')
    for f in stale:
        f.unlink()

## Compute Topology for Priority CPC Section Pairs

10 priority pairs spanning the major cross-disciplinary interfaces where
breakthroughs tend to occur. Each window (~260×260 distance matrix) completes
in seconds.

In [ ]:
# %% Define pair descriptions for labels
PAIR_DESCRIPTIONS = {
    'AxC': 'Biotech / Pharma',
    'AxG': 'Medical Devices',
    'CxG': 'Materials / Sensors',
    'CxH': 'Batteries / Energy',
    'GxH': 'Semiconductors / Computing',
    'BxG': 'Manufacturing Tech',
    'BxH': 'Automation / Robotics',
    'AxH': 'Health Tech / Wearables',
    'CxB': 'Chemical Engineering',
    'FxH': 'Electromechanical / Power',
}

print(f'Priority pairs: {len(PRIORITY_PAIRS)}')
for (a, b) in PRIORITY_PAIRS:
    key = f'{a}x{b}'
    print(f'  {key}: {PAIR_DESCRIPTIONS.get(key, "")}')

In [ ]:
# %% Compute topology for all 10 priority pairs (with incremental caching)
pair_results = compute_all_priority_pairs(
    citations, cpc_map,
    cache_dir=str(DATA_DIR / 'topology_cache'),
    start_year=1985,
    end_year=2023,
)

print(f'\nTotal observations: {len(pair_results)}')
print(f'Pairs computed: {pair_results["section_pair"].nunique()}')

# Split into per-pair dict for easy access
topology_results = {pair: group.reset_index(drop=True) for pair, group in pair_results.groupby('section_pair')}

gc.collect()

In [ ]:
# %% Compute global topology (all CPC sections combined)
global_results = compute_global_topology(
    citations, cpc_map,
    cache_dir=str(DATA_DIR / 'topology_cache'),
    start_year=1985,
    end_year=2023,
)

print(f'Global windows: {len(global_results)}')
if len(global_results) > 0:
    print(f'Global β₁ range: {global_results["beta_1"].min()} - {global_results["beta_1"].max()}')
    print(f'Global PE range: {global_results["persistence_entropy"].min():.3f} - {global_results["persistence_entropy"].max():.3f}')

## Figure 1: β₁ Time Series for Each CPC Pair

β₁ counts 1-dimensional loops in the knowledge space — circular citation
flows between clusters of technology fields. When β₁ rises, fields that
were previously citing each other in a tree-like hierarchy start forming
circular exchange patterns.

In [ ]:
# %% Figure 1: β₁ time series for 10 priority pairs
fig, axes = plt.subplots(2, 5, figsize=(24, 10), sharex=True)
axes = axes.flatten()

pair_keys = sorted(topology_results.keys())

for i, pair_key in enumerate(pair_keys):
    ax = axes[i]
    df = topology_results[pair_key]
    
    ax.plot(df['window_end'], df['beta_1'], color=PALETTE_LIST[i % len(PALETTE_LIST)], linewidth=2)
    ax.fill_between(df['window_end'], 0, df['beta_1'], alpha=0.15, color=PALETTE_LIST[i % len(PALETTE_LIST)])
    
    desc = PAIR_DESCRIPTIONS.get(pair_key, pair_key)
    ax.set_title(f'{pair_key}\n{desc}', fontsize=10)
    ax.set_ylabel('β₁')
    year_axis(ax, start=1985, end=2023)

# Hide unused axes
for j in range(len(pair_keys), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 1: β₁ (1-Dimensional Loops) in the Knowledge Space Over Time', fontsize=16, y=1.02)
fig.tight_layout()
save_figure(fig, '02_beta1_time_series')

## Figure 2: Global Knowledge Landscape — β₁ and Persistence Entropy

The full ~260-point CPC subclass distance matrix gives us the topology of
the *entire* patent knowledge landscape, not filtered to any pair.

In [ ]:
# %% Figure 2: Global β₁ and persistence entropy
if len(global_results) > 0:
    fig, ax1 = plt.subplots(figsize=(12, 7))

    ax1.plot(global_results['window_end'], global_results['beta_1'],
             color=PALETTE['red'], linewidth=2.5, label='β₁ (loops)')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('β₁', color=PALETTE['red'])
    ax1.tick_params(axis='y', labelcolor=PALETTE['red'])
    year_axis(ax1, start=1985, end=2023)

    ax2 = ax1.twinx()
    ax2.plot(global_results['window_end'], global_results['persistence_entropy'],
             color=PALETTE['blue'], linewidth=2.5, linestyle='--', label='Persistence Entropy')
    ax2.set_ylabel('Persistence Entropy (bits)', color=PALETTE['blue'])
    ax2.tick_params(axis='y', labelcolor=PALETTE['blue'])

    # Combine legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    fig.suptitle('Figure 2: Global Knowledge Landscape Topology Over Time', fontsize=14)
    fig.tight_layout()
    save_figure(fig, '02_global_topology')

## Figure 3: β₁ Heatmap Across All Pairs

Where is topology changing fastest? This heatmap shows β₁ across all
10 priority pairs over time.

In [ ]:
# %% Figure 3: β₁ heatmap
# Build matrix: pairs x years
all_years = sorted(pair_results['window_end'].unique())
heatmap_data = []

for pair_key in pair_keys:
    df = topology_results[pair_key]
    year_to_b1 = dict(zip(df['window_end'], df['beta_1']))
    row = [year_to_b1.get(y, np.nan) for y in all_years]
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(
    heatmap_data,
    index=[f'{k} ({PAIR_DESCRIPTIONS.get(k, "")})' for k in pair_keys],
    columns=all_years,
)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df, ax=ax, cmap='YlOrRd', linewidths=0.5,
    cbar_kws={'label': 'β₁ (number of 1-dimensional loops)'},
)
ax.set_xlabel('Window End Year')
ax.set_ylabel('CPC Section Pair')
ax.set_title('Figure 3: β₁ Across CPC Pairs Over Time — Where Is Topology Changing?')

# Show every 5th year label
xtick_positions = [i for i, y in enumerate(all_years) if y % 5 == 0]
xtick_labels = [str(all_years[i]) for i in xtick_positions]
ax.set_xticks(xtick_positions)
ax.set_xticklabels(xtick_labels, rotation=45)

fig.tight_layout()
save_figure(fig, '02_beta1_heatmap')

## Figure 4: Persistence Diagram Gallery

Persistence diagrams are the raw output of persistent homology. Each point
represents a topological feature (connected component, loop, or void).
Points far from the diagonal are long-lived, persistent structures.

We compare 3 "interesting" windows (high β₁) against 3 "boring" windows
(low β₁) to see what the topology looks like when cross-field activity spikes.

In [ ]:
# %% Figure 4: Persistence diagram gallery
# Pick a pair with clear variation for the gallery
gallery_pair = pair_keys[0]  # Use first pair
gallery_df = topology_results[gallery_pair]

# Sort by β₁ to find interesting vs boring windows
sorted_by_b1 = gallery_df.sort_values('beta_1')

# 3 lowest β₁ (boring) and 3 highest β₁ (interesting)
boring_windows = sorted_by_b1.head(3)
interesting_windows = sorted_by_b1.tail(3)
selected = pd.concat([boring_windows, interesting_windows])

# Filter cpc_map to this pair's sections
pair_sections = gallery_pair.split('x')
pair_cpc = cpc_map[cpc_map['cpc_section'].isin(pair_sections)]
pair_patents = set(pair_cpc['patent_id'].unique())
pair_citations = citations[
    citations['citing_id'].isin(pair_patents) | citations['cited_id'].isin(pair_patents)
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (_, row) in enumerate(selected.iterrows()):
    ax = axes[idx // 3, idx % 3]
    ws, we = int(row['window_start']), int(row['window_end'])
    b1_val = int(row['beta_1'])
    
    # Compute persistence diagram for this window
    cocite_df, labels = build_cocitation_matrix(pair_citations, pair_cpc, ws, we, level='subclass')
    
    if cocite_df.empty or len(labels) < 3:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)
        continue
    
    dist_matrix, active_mask = cocitation_to_distance(cocite_df.values)
    if dist_matrix.size == 0:
        ax.text(0.5, 0.5, 'No active classes', ha='center', va='center', transform=ax.transAxes)
        continue
    
    diagrams = compute_persistence(dist_matrix, max_dim=1)
    
    # Plot H0 (blue) and H1 (red)
    for dim, (dgm, color, label) in enumerate([
        (diagrams[0], PALETTE['blue'], 'H₀ (components)'),
        (diagrams[1] if len(diagrams) > 1 else np.array([]).reshape(0, 2), PALETTE['red'], 'H₁ (loops)'),
    ]):
        if len(dgm) > 0:
            finite = dgm[np.isfinite(dgm[:, 1])]
            if len(finite) > 0:
                ax.scatter(finite[:, 0], finite[:, 1], c=color, s=30, alpha=0.7, label=label, zorder=3)
    
    # Diagonal line
    lims = ax.get_xlim()
    ax.plot([0, 2], [0, 2], 'k--', alpha=0.3, linewidth=1)
    
    category = 'LOW β₁' if idx < 3 else 'HIGH β₁'
    ax.set_title(f'{ws}-{we} (β₁={b1_val}, {category})', fontsize=10)
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')
    if idx == 0 or idx == 3:
        ax.legend(fontsize=8)

fig.suptitle(f'Figure 4: Persistence Diagrams — {gallery_pair} ({PAIR_DESCRIPTIONS.get(gallery_pair, "")})', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '02_persistence_diagrams')

# Cleanup
del pair_citations, pair_cpc, pair_patents
gc.collect()

## Figure 5: CPC Mixing Rate vs β₁

The CPC mixing rate (fraction of citations crossing section boundaries) is
a simple metric from Notebook 01. Does β₁ (a topological metric) track it
or diverge? If they diverge, topology captures information that simple
cross-citation counting misses.

In [ ]:
# %% Figure 5: CPC mixing rate vs β₁
mixing = cpc_mixing_rate(citations, cpc_map)

# Compute mean β₁ across all 10 pairs per year
mean_beta1 = pair_results.groupby('window_end')['beta_1'].mean().reset_index()

fig, ax1 = plt.subplots(figsize=(12, 7))

# Mixing rate
ax1.plot(mixing['year'], mixing['mixing_rate'], color=PALETTE['blue'], linewidth=2.5, label='CPC Mixing Rate')
ax1.set_xlabel('Year')
ax1.set_ylabel('CPC Mixing Rate', color=PALETTE['blue'])
ax1.tick_params(axis='y', labelcolor=PALETTE['blue'])
year_axis(ax1, start=1985, end=2023)

# Mean β₁
ax2 = ax1.twinx()
ax2.plot(mean_beta1['window_end'], mean_beta1['beta_1'],
         color=PALETTE['red'], linewidth=2.5, linestyle='--', label='Mean β₁ (10 pairs)')
ax2.set_ylabel('Mean β₁', color=PALETTE['red'])
ax2.tick_params(axis='y', labelcolor=PALETTE['red'])

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.suptitle('Figure 5: Simple Metric (Mixing Rate) vs Topological Metric (β₁)', fontsize=14)
fig.tight_layout()
save_figure(fig, '02_beta1_vs_mixing')

## Summary

### Methodology

This notebook uses **CPC subclass co-citation distance matrices** rather than
clique complexes on raw citation subgraphs. Each ~260-point distance matrix
completes in seconds, making the full analysis feasible on standard hardware.

### Key Observations

- **β₁ time series** reveal when circular citation flows form between technology
  clusters — a structural precursor to cross-disciplinary convergence.
- **Persistence entropy** measures the diversity of topological features. Higher
  entropy suggests a more complex, multi-scale knowledge landscape.
- **Comparison with mixing rate**: If β₁ tracks the mixing rate closely, topology
  may be redundant with simpler metrics. If they diverge, topology captures
  structural information (loops, voids) that counting cross-citations misses.

### What Comes Next

These topological time series feed into:
- **Notebook 04** (Precursor Test): Do β₁ spikes systematically precede known breakthroughs?
- **Notebook 05** (Predictability Horizon): Can topological features forecast where breakthroughs land?